In [1]:
from pathlib import Path
import sys
import json
import time
import pandas as pd
import numpy as np
import xgboost as xgb

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "Fraud-detection-ML-V2" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
SRC_DIR = PROJECT_ROOT / "src"

TEST_FEATURE_PATH = PROCESSED_DATA_DIR / "ml_test_features.csv"
MODEL_PATH = MODELS_DIR / "xgboost_fraud_detector.json"
RULE_CONFIG_PATH = MODELS_DIR / "rule_engine_config.json"
DECISION_CONFIG_PATH = MODELS_DIR / "decision_engine_config.json"
FEATURE_CONTRACT_PATH = MODELS_DIR / "feature_columns.json"

required_paths = [
    TEST_FEATURE_PATH,
    MODEL_PATH,
    RULE_CONFIG_PATH,
    DECISION_CONFIG_PATH,
    FEATURE_CONTRACT_PATH
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing required integration files:\n"
        + "\n".join(missing_paths)
    )

df = pd.read_csv(TEST_FEATURE_PATH)

with open(RULE_CONFIG_PATH, "r") as file:
    rule_config = json.load(file)

with open(DECISION_CONFIG_PATH, "r") as file:
    decision_config = json.load(file)

with open(FEATURE_CONTRACT_PATH, "r") as file:
    feature_contract = json.load(file)

FEATURE_COLUMNS = feature_contract["features"]

model = xgb.XGBClassifier()
model.load_model(MODEL_PATH)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rule_engine import evaluate_rules
from decision_engine import make_decision

print("========== PHASE 11 INITIALIZATION ==========")
print("Test rows:", len(df))
print("Model loaded:", True)
print("Rule engine loaded:", callable(evaluate_rules))
print("Decision engine loaded:", callable(make_decision))
print("Feature count:", len(FEATURE_COLUMNS))
print("=============================================")

========== PHASE 11 INITIALIZATION ==========
Test rows: 555719
Model loaded: True
Rule engine loaded: True
Decision engine loaded: True
Feature count: 6


In [2]:
required_input_columns = [
    "transaction_id",
    "user_id",
    *FEATURE_COLUMNS,
    "is_fraud"
]

missing_input_columns = [
    column
    for column in required_input_columns
    if column not in df.columns
]

if missing_input_columns:
    raise ValueError(
        f"Missing integration input columns: {missing_input_columns}"
    )

for column in FEATURE_COLUMNS:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

if df[FEATURE_COLUMNS].isnull().any().any():
    raise ValueError(
        "Integration features contain missing values."
    )

if not np.isfinite(
    df[FEATURE_COLUMNS].to_numpy(dtype=float)
).all():
    raise ValueError(
        "Integration features contain non-finite values."
    )

print("========== INPUT CONTRACT VALIDATION ==========")
print("All required fields present:", True)
print("Feature count:", len(FEATURE_COLUMNS))
print("Feature values valid:", True)
print("===============================================")

========== INPUT CONTRACT VALIDATION ==========
All required fields present: True
Feature count: 6
Feature values valid: True


In [3]:
import shap

explainer = shap.TreeExplainer(
    model
)

print("========== SHAP EXPLAINER ==========")
print("SHAP imported:", True)
print("TreeExplainer created:", True)
print("====================================")

========== SHAP EXPLAINER ==========
SHAP imported: True
TreeExplainer created: True


In [4]:
SHAP_EXPLAINER_CODE = '''from pathlib import Path
import numpy as np
import pandas as pd
import shap
import xgboost as xgb

SRC_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = SRC_DIR.parent
MODELS_DIR = PROJECT_ROOT / "models"

MODEL_PATH = MODELS_DIR / "xgboost_fraud_detector.json"
FEATURE_PATH = MODELS_DIR / "feature_columns.json"

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model file not found: {MODEL_PATH}"
    )

if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Feature contract file not found: {FEATURE_PATH}"
    )

with open(FEATURE_PATH, "r", encoding="utf-8") as file:
    FEATURE_CONFIG = __import__("json").load(file)

FEATURE_COLUMNS = FEATURE_CONFIG["features"]

MODEL = xgb.XGBClassifier()
MODEL.load_model(MODEL_PATH)

EXPLAINER = shap.TreeExplainer(MODEL)


def explain_transaction(transaction, top_n=3):
    if not isinstance(transaction, dict):
        raise TypeError(
            "Transaction must be a dictionary."
        )

    missing_features = [
        feature
        for feature in FEATURE_COLUMNS
        if feature not in transaction
    ]

    if missing_features:
        raise ValueError(
            f"Missing SHAP features: {missing_features}"
        )

    input_df = pd.DataFrame(
        [
            {
                feature: transaction[feature]
                for feature in FEATURE_COLUMNS
            }
        ],
        columns=FEATURE_COLUMNS
    )

    for feature in FEATURE_COLUMNS:
        input_df[feature] = pd.to_numeric(
            input_df[feature],
            errors="coerce"
        )

    if input_df.isnull().any().any():
        raise ValueError(
            "Invalid SHAP input values."
        )

    input_array = input_df.to_numpy(dtype=float)

    if not np.isfinite(input_array).all():
        raise ValueError(
            "Non-finite SHAP input values."
        )

    values = np.asarray(
        EXPLAINER.shap_values(input_df)
    )

    if values.ndim == 2:
        values = values[0]

    if len(values) != len(FEATURE_COLUMNS):
        raise ValueError(
            "SHAP output size does not match feature count."
        )

    explanation_df = pd.DataFrame({
        "feature": FEATURE_COLUMNS,
        "feature_value": input_df.iloc[0].to_numpy(),
        "shap_value": values
    })

    explanation_df["absolute_shap"] = (
        explanation_df["shap_value"].abs()
    )

    explanation_df = (
        explanation_df
        .sort_values(
            "absolute_shap",
            ascending=False
        )
        .reset_index(drop=True)
    )

    top_features = explanation_df.head(
        max(1, int(top_n))
    )

    return {
        "top_features": top_features[
            [
                "feature",
                "feature_value",
                "shap_value"
            ]
        ].to_dict(orient="records")
    }
'''

SHAP_EXPLAINER_PATH = SRC_DIR / "shap_explainer.py"

SHAP_EXPLAINER_PATH.write_text(
    SHAP_EXPLAINER_CODE,
    encoding="utf-8"
)

print("========== SHAP MODULE ==========")
print("File:", SHAP_EXPLAINER_PATH)
print("File exists:", SHAP_EXPLAINER_PATH.exists())
print("=================================")

========== SHAP MODULE ==========
File: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\src\shap_explainer.py
File exists: True


In [5]:
from shap_explainer import explain_transaction

sample_transaction = (
    df.iloc[0][FEATURE_COLUMNS]
    .to_dict()
)

sample_shap_result = explain_transaction(
    sample_transaction,
    top_n=3
)

print("========== PRODUCTION SHAP TEST ==========")
print("SHAP result:")
print(sample_shap_result)
print()
print(
    "Top SHAP features:",
    len(sample_shap_result["top_features"])
)
print("==========================================")

========== PRODUCTION SHAP TEST ==========
SHAP result:
{'top_features': [{'feature': 'amount', 'feature_value': 124.66, 'shap_value': -1.4020209312438965}, {'feature': 'time_since_last_txn_sec', 'feature_value': 15081.0, 'shap_value': -0.650715708732605}, {'feature': 'amount_vs_avg_ratio', 'feature_value': 2.2251429768799413, 'shap_value': -0.46119046211242676}]}

Top SHAP features: 3


In [6]:
from ml_service import get_ml_score
from shap_explainer import explain_transaction

def detect_transaction(transaction):
    required_fields = [
        "transaction_id",
        "user_id",
        *FEATURE_COLUMNS
    ]

    missing_fields = [
        field
        for field in required_fields
        if field not in transaction
    ]

    if missing_fields:
        raise ValueError(
            f"Missing detection fields: {missing_fields}"
        )

    feature_transaction = {
        feature: transaction[feature]
        for feature in FEATURE_COLUMNS
    }

    ml_result = get_ml_score(
        feature_transaction
    )

    rule_result = evaluate_rules(
        feature_transaction
    )

    decision_result = make_decision(
        ml_result["ml_score"],
        rule_result
    )

    shap_result = explain_transaction(
        feature_transaction,
        top_n=3
    )

    return {
        "transaction_id": transaction["transaction_id"],
        "user_id": transaction["user_id"],
        "ml_score": ml_result["ml_score"],
        "ml_prediction": ml_result["ml_prediction"],
        "ml_label": ml_result["ml_label"],
        "rule_flags": decision_result["rule_flags"],
        "rule_score": rule_result["rule_score"],
        "risk_score": decision_result["risk_score"],
        "decision": decision_result["decision"],
        "shap": shap_result
    }

print("========== INTEGRATED DETECTION ==========")
print(
    "Detection function created:",
    callable(detect_transaction)
)
print("==========================================")

========== INTEGRATED DETECTION ==========
Detection function created: True


In [7]:
def feature_reason_text(feature, value, shap_value):
    direction = (
        "increased"
        if float(shap_value) > 0
        else "decreased"
    )

    if feature == "amount":
        value_text = f"amount={float(value):.2f}"

    elif feature == "amount_vs_avg_ratio":
        value_text = (
            f"amount_vs_avg_ratio={float(value):.2f}"
        )

    elif feature == "txn_count_last_5min":
        value_text = (
            f"txn_count_last_5min={int(value)}"
        )

    elif feature == "time_since_last_txn_sec":
        value_text = (
            f"time_since_last_txn_sec={float(value):.2f}"
        )

    elif feature == "distance_from_last_location_km":
        value_text = (
            f"distance_from_last_location_km={float(value):.2f}"
        )

    elif feature == "merchant_category_is_new_for_user":
        value_text = (
            f"merchant_category_is_new_for_user={int(value)}"
        )

    else:
        value_text = f"{feature}={value}"

    return f"{value_text} {direction} fraud risk"


def generate_integrated_reason(
    rule_flags,
    ml_score,
    shap_result,
    decision
):
    parts = []

    if rule_flags:
        parts.append(
            "Triggered rules: "
            + ", ".join(rule_flags)
        )
    else:
        parts.append(
            "No deterministic rules triggered"
        )

    parts.append(
        f"ML fraud score: {float(ml_score):.4f}"
    )

    for item in shap_result["top_features"]:
        parts.append(
            feature_reason_text(
                item["feature"],
                item["feature_value"],
                item["shap_value"]
            )
        )

    parts.append(
        f"Decision: {decision.upper()}"
    )

    return ". ".join(parts) + "."

print("========== EXPLANATION FUNCTION ==========")
print(
    "Explanation function created:",
    callable(generate_integrated_reason)
)
print("==========================================")

========== EXPLANATION FUNCTION ==========
Explanation function created: True


In [8]:
real_transaction = (
    df.iloc[0]
    [
        [
            "transaction_id",
            "user_id",
            *FEATURE_COLUMNS
        ]
    ]
    .to_dict()
)

integrated_result = detect_transaction(
    real_transaction
)

integrated_result[
    "human_readable_reason"
] = generate_integrated_reason(
    integrated_result["rule_flags"],
    integrated_result["ml_score"],
    integrated_result["shap"],
    integrated_result["decision"]
)

print("========== COMPLETE DETECTION TEST ==========")
print(
    "Transaction ID:",
    integrated_result["transaction_id"]
)

print(
    "User ID:",
    integrated_result["user_id"]
)

print(
    "ML score:",
    integrated_result["ml_score"]
)

print(
    "Rule flags:",
    integrated_result["rule_flags"]
)

print(
    "Risk score:",
    integrated_result["risk_score"]
)

print(
    "Decision:",
    integrated_result["decision"]
)

print(
    "SHAP:",
    integrated_result["shap"]
)

print(
    "Human-readable reason:",
    integrated_result["human_readable_reason"]
)

print("============================================")

========== COMPLETE DETECTION TEST ==========
Transaction ID: TST_000000001
User ID: 60416207185
ML score: 0.07883047312498093
Rule flags: []
Risk score: 0.03941523656249046
Decision: allow
SHAP: {'top_features': [{'feature': 'amount', 'feature_value': 124.66, 'shap_value': -1.4020209312438965}, {'feature': 'time_since_last_txn_sec', 'feature_value': 15081.0, 'shap_value': -0.650715708732605}, {'feature': 'amount_vs_avg_ratio', 'feature_value': 2.2251429768799413, 'shap_value': -0.46119046211242676}]}
Human-readable reason: No deterministic rules triggered. ML fraud score: 0.0788. amount=124.66 decreased fraud risk. time_since_last_txn_sec=15081.00 decreased fraud risk. amount_vs_avg_ratio=2.23 decreased fraud risk. Decision: ALLOW.


In [9]:
from datetime import datetime, timezone

def create_scoring_output(
    transaction,
    detection_result,
    processing_latency_ms
):
    return {
        "transaction_id": transaction["transaction_id"],
        "user_id": transaction["user_id"],
        "risk_score": float(
            detection_result["risk_score"]
        ),
        "rule_flags": list(
            detection_result["rule_flags"]
        ),
        "ml_fraud_score": float(
            detection_result["ml_score"]
        ),
        "decision": detection_result["decision"],
        "human_readable_reason": (
            detection_result[
                "human_readable_reason"
            ]
        ),
        "processed_at": datetime.now(
            timezone.utc
        ).isoformat(),
        "latency_ms": float(
            processing_latency_ms
        )
    }

start_time = time.perf_counter()

temporary_result = detect_transaction(
    real_transaction
)

temporary_result[
    "human_readable_reason"
] = generate_integrated_reason(
    temporary_result["rule_flags"],
    temporary_result["ml_score"],
    temporary_result["shap"],
    temporary_result["decision"]
)

processing_latency_ms = (
    time.perf_counter()
    - start_time
) * 1000

scoring_output = create_scoring_output(
    real_transaction,
    temporary_result,
    processing_latency_ms
)

print("========== SCORING OUTPUT ==========")
print(
    json.dumps(
        scoring_output,
        indent=4
    )
)
print("====================================")

========== SCORING OUTPUT ==========
{
    "transaction_id": "TST_000000001",
    "user_id": 60416207185,
    "risk_score": 0.03941523656249046,
    "rule_flags": [],
    "ml_fraud_score": 0.07883047312498093,
    "decision": "allow",
    "human_readable_reason": "No deterministic rules triggered. ML fraud score: 0.0788. amount=124.66 decreased fraud risk. time_since_last_txn_sec=15081.00 decreased fraud risk. amount_vs_avg_ratio=2.23 decreased fraud risk. Decision: ALLOW.",
    "processed_at": "2026-09-05T12:14:56.126988+00:00",
    "latency_ms": 23.80449999964185
}


In [10]:
required_output_fields = [
    "transaction_id",
    "user_id",
    "risk_score",
    "rule_flags",
    "ml_fraud_score",
    "decision",
    "human_readable_reason",
    "processed_at",
    "latency_ms"
]

missing_output_fields = [
    field
    for field in required_output_fields
    if field not in scoring_output
]

risk_score_valid = (
    0 <= scoring_output["risk_score"] <= 1
)

ml_score_valid = (
    0 <= scoring_output["ml_fraud_score"] <= 1
)

decision_valid = (
    scoring_output["decision"]
    in [
        "allow",
        "otp",
        "review",
        "block"
    ]
)

rule_flags_valid = isinstance(
    scoring_output["rule_flags"],
    list
)

reason_valid = (
    isinstance(
        scoring_output["human_readable_reason"],
        str
    )
    and
    len(
        scoring_output[
            "human_readable_reason"
        ].strip()
    ) > 0
)

latency_valid = (
    scoring_output["latency_ms"] >= 0
)

print("========== OUTPUT CONTRACT VALIDATION ==========")

print(
    "All required fields present:",
    len(missing_output_fields) == 0
)

print(
    "Risk score valid:",
    risk_score_valid
)

print(
    "ML fraud score valid:",
    ml_score_valid
)

print(
    "Decision valid:",
    decision_valid
)

print(
    "Rule flags valid:",
    rule_flags_valid
)

print(
    "Human-readable reason valid:",
    reason_valid
)

print(
    "Latency valid:",
    latency_valid
)

print(
    "Overall contract valid:",
    all([
        len(missing_output_fields) == 0,
        risk_score_valid,
        ml_score_valid,
        decision_valid,
        rule_flags_valid,
        reason_valid,
        latency_valid
    ])
)

print("=================================================")

========== OUTPUT CONTRACT VALIDATION ==========
All required fields present: True
Risk score valid: True
ML fraud score valid: True
Decision valid: True
Rule flags valid: True
Human-readable reason valid: True
Latency valid: True
Overall contract valid: True


In [11]:
integration_results = []

sample_size = min(
    100,
    len(df)
)

for _, row in df.head(sample_size).iterrows():

    transaction = (
        row[
            [
                "transaction_id",
                "user_id",
                *FEATURE_COLUMNS
            ]
        ]
        .to_dict()
    )

    start_time = time.perf_counter()

    result = detect_transaction(
        transaction
    )

    result[
        "human_readable_reason"
    ] = generate_integrated_reason(
        result["rule_flags"],
        result["ml_score"],
        result["shap"],
        result["decision"]
    )

    latency_ms = (
        time.perf_counter()
        - start_time
    ) * 1000

    integration_results.append(
        create_scoring_output(
            transaction,
            result,
            latency_ms
        )
    )

integration_results_df = pd.DataFrame(
    integration_results
)

print("========== 100 TRANSACTION INTEGRATION ==========")
print(
    "Transactions processed:",
    len(integration_results_df)
)

print(
    "===============================================")

========== 100 TRANSACTION INTEGRATION ==========
Transactions processed: 100


In [12]:
batch_risk_valid = (
    integration_results_df[
        "risk_score"
    ]
    .between(0, 1)
    .all()
)

batch_ml_valid = (
    integration_results_df[
        "ml_fraud_score"
    ]
    .between(0, 1)
    .all()
)

batch_decision_valid = (
    integration_results_df[
        "decision"
    ]
    .isin([
        "allow",
        "otp",
        "review",
        "block"
    ])
    .all()
)

batch_reason_valid = all(
    isinstance(reason, str)
    and len(reason.strip()) > 0
    for reason in integration_results_df[
        "human_readable_reason"
    ]
)

batch_latency_valid = (
    integration_results_df[
        "latency_ms"
    ] >= 0
).all()

print("========== 100 TRANSACTION VALIDATION ==========")

print(
    "All risk scores valid:",
    batch_risk_valid
)

print(
    "All ML scores valid:",
    batch_ml_valid
)

print(
    "All decisions valid:",
    batch_decision_valid
)

print(
    "All explanations valid:",
    batch_reason_valid
)

print(
    "All latency values valid:",
    batch_latency_valid
)

print(
    "Overall batch validation:",
    all([
        len(integration_results_df) == sample_size,
        batch_risk_valid,
        batch_ml_valid,
        batch_decision_valid,
        batch_reason_valid,
        batch_latency_valid
    ])
)

print("==============================================")

========== 100 TRANSACTION VALIDATION ==========
All risk scores valid: True
All ML scores valid: True
All decisions valid: True
All explanations valid: True
All latency values valid: True
Overall batch validation: True


In [13]:
DETECTION_SERVICE_CODE = '''from datetime import datetime, timezone
import time

from ml_service import get_ml_score
from rule_engine import evaluate_rules
from shap_explainer import explain_transaction
from decision_engine import make_decision


FEATURE_COLUMNS = [
    "amount",
    "amount_vs_avg_ratio",
    "txn_count_last_5min",
    "time_since_last_txn_sec",
    "distance_from_last_location_km",
    "merchant_category_is_new_for_user"
]


def feature_reason_text(
    feature,
    value,
    shap_value
):
    direction = (
        "increased"
        if float(shap_value) > 0
        else "decreased"
    )

    if feature == "amount":
        value_text = f"amount={float(value):.2f}"
    elif feature == "amount_vs_avg_ratio":
        value_text = (
            f"amount_vs_avg_ratio={float(value):.2f}"
        )
    elif feature == "txn_count_last_5min":
        value_text = (
            f"txn_count_last_5min={int(value)}"
        )
    elif feature == "time_since_last_txn_sec":
        value_text = (
            f"time_since_last_txn_sec={float(value):.2f}"
        )
    elif feature == "distance_from_last_location_km":
        value_text = (
            f"distance_from_last_location_km={float(value):.2f}"
        )
    elif feature == "merchant_category_is_new_for_user":
        value_text = (
            f"merchant_category_is_new_for_user={int(value)}"
        )
    else:
        value_text = f"{feature}={value}"

    return (
        f"{value_text} {direction} fraud risk"
    )


def generate_reason(
    rule_flags,
    ml_score,
    shap_result,
    decision
):
    parts = []

    if rule_flags:
        parts.append(
            "Triggered rules: "
            + ", ".join(rule_flags)
        )
    else:
        parts.append(
            "No deterministic rules triggered"
        )

    parts.append(
        f"ML fraud score: {float(ml_score):.4f}"
    )

    for item in shap_result["top_features"]:
        parts.append(
            feature_reason_text(
                item["feature"],
                item["feature_value"],
                item["shap_value"]
            )
        )

    parts.append(
        f"Decision: {decision.upper()}"
    )

    return ". ".join(parts) + "."


def score_transaction(transaction):
    start_time = time.perf_counter()

    required_fields = [
        "transaction_id",
        "user_id",
        *FEATURE_COLUMNS
    ]

    missing_fields = [
        field
        for field in required_fields
        if field not in transaction
    ]

    if missing_fields:
        raise ValueError(
            f"Missing detection fields: {missing_fields}"
        )

    features = {
        feature: transaction[feature]
        for feature in FEATURE_COLUMNS
    }

    ml_result = get_ml_score(
        features
    )

    rule_result = evaluate_rules(
        features
    )

    decision_result = make_decision(
        ml_result["ml_score"],
        rule_result
    )

    shap_result = explain_transaction(
        features,
        top_n=3
    )

    reason = generate_reason(
        decision_result["rule_flags"],
        ml_result["ml_score"],
        shap_result,
        decision_result["decision"]
    )

    latency_ms = (
        time.perf_counter()
        - start_time
    ) * 1000

    return {
        "transaction_id": transaction["transaction_id"],
        "user_id": transaction["user_id"],
        "risk_score": float(
            decision_result["risk_score"]
        ),
        "rule_flags": list(
            decision_result["rule_flags"]
        ),
        "ml_fraud_score": float(
            ml_result["ml_score"]
        ),
        "decision": decision_result["decision"],
        "human_readable_reason": reason,
        "processed_at": datetime.now(
            timezone.utc
        ).isoformat(),
        "latency_ms": float(
            latency_ms
        )
    }
'''

DETECTION_SERVICE_PATH = SRC_DIR / "detection_service.py"

DETECTION_SERVICE_PATH.write_text(
    DETECTION_SERVICE_CODE,
    encoding="utf-8"
)

print("========== DETECTION SERVICE ==========")
print("File:", DETECTION_SERVICE_PATH)
print("File exists:", DETECTION_SERVICE_PATH.exists())
print("=======================================")

========== DETECTION SERVICE ==========
File: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\src\detection_service.py
File exists: True


In [14]:
from detection_service import score_transaction

production_output = score_transaction(
    real_transaction
)

print("========== PRODUCTION DETECTION TEST ==========")
print(
    json.dumps(
        production_output,
        indent=4
    )
)
print("===============================================")

========== PRODUCTION DETECTION TEST ==========
{
    "transaction_id": "TST_000000001",
    "user_id": 60416207185,
    "risk_score": 0.03941523656249046,
    "rule_flags": [],
    "ml_fraud_score": 0.07883047312498093,
    "decision": "allow",
    "human_readable_reason": "No deterministic rules triggered. ML fraud score: 0.0788. amount=124.66 decreased fraud risk. time_since_last_txn_sec=15081.00 decreased fraud risk. amount_vs_avg_ratio=2.23 decreased fraud risk. Decision: ALLOW.",
    "processed_at": "2026-09-05T12:20:51.985698+00:00",
    "latency_ms": 20.50150000013673
}


In [15]:
production_output_valid = all([
    all(
        field in production_output
        for field in required_output_fields
    ),
    0 <= production_output["risk_score"] <= 1,
    0 <= production_output["ml_fraud_score"] <= 1,
    production_output["decision"] in [
        "allow",
        "otp",
        "review",
        "block"
    ],
    isinstance(
        production_output["rule_flags"],
        list
    ),
    isinstance(
        production_output["human_readable_reason"],
        str
    ),
    len(
        production_output[
            "human_readable_reason"
        ].strip()
    ) > 0,
    production_output["latency_ms"] >= 0
])

print("========== PRODUCTION OUTPUT VALIDATION ==========")
print(
    "Production output valid:",
    production_output_valid
)
print("===================================================")

========== PRODUCTION OUTPUT VALIDATION ==========
Production output valid: True


In [16]:
production_batch = []

for _, row in df.head(sample_size).iterrows():

    transaction = (
        row[
            [
                "transaction_id",
                "user_id",
                *FEATURE_COLUMNS
            ]
        ]
        .to_dict()
    )

    production_batch.append(
        score_transaction(
            transaction
        )
    )

production_batch_df = pd.DataFrame(
    production_batch
)

production_batch_valid = all([
    len(production_batch_df) == sample_size,
    production_batch_df[
        "risk_score"
    ].between(0, 1).all(),
    production_batch_df[
        "ml_fraud_score"
    ].between(0, 1).all(),
    production_batch_df[
        "decision"
    ].isin([
        "allow",
        "otp",
        "review",
        "block"
    ]).all(),
    (
        production_batch_df[
            "latency_ms"
        ] >= 0
    ).all()
])

print("========== PRODUCTION 100 TRANSACTION TEST ==========")
print(
    "Transactions processed:",
    len(production_batch_df)
)

print(
    "All production outputs valid:",
    production_batch_valid
)

print()
print("Decision counts:")
print(
    production_batch_df[
        "decision"
    ].value_counts()
)

print("=====================================================")

========== PRODUCTION 100 TRANSACTION TEST ==========
Transactions processed: 100
All production outputs valid: True

Decision counts:
decision
allow    96
otp       4
Name: count, dtype: int64


In [17]:
integration_validation = {
    "feature_count": len(FEATURE_COLUMNS),
    "rule_engine_available": callable(evaluate_rules),
    "ml_service_available": callable(get_ml_score),
    "shap_service_available": callable(explain_transaction),
    "decision_engine_available": callable(make_decision),
    "single_transaction_output_valid": all([
        len(missing_output_fields) == 0,
        risk_score_valid,
        ml_score_valid,
        decision_valid,
        rule_flags_valid,
        reason_valid,
        latency_valid
    ]),
    "100_transaction_integration_valid": all([
        len(integration_results_df) == sample_size,
        batch_risk_valid,
        batch_ml_valid,
        batch_decision_valid,
        batch_reason_valid,
        batch_latency_valid
    ]),
    "production_output_valid": production_output_valid,
    "production_batch_valid": production_batch_valid,
    "scoring_output_fields": required_output_fields
}

INTEGRATION_VALIDATION_PATH = (
    METRICS_DIR
    / "rule_ml_shap_integration_validation.json"
)

with open(
    INTEGRATION_VALIDATION_PATH,
    "w"
) as file:
    json.dump(
        integration_validation,
        file,
        indent=4
    )

print("========== INTEGRATION VALIDATION SAVED ==========")
print(
    "File:",
    INTEGRATION_VALIDATION_PATH
)

print(
    "File exists:",
    INTEGRATION_VALIDATION_PATH.exists()
)

print("==================================================")

========== INTEGRATION VALIDATION SAVED ==========
File: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\metrics\rule_ml_shap_integration_validation.json
File exists: True


In [18]:
final_integration_ready = all([
    callable(evaluate_rules),
    callable(get_ml_score),
    callable(explain_transaction),
    callable(make_decision),
    SHAP_EXPLAINER_PATH.exists(),
    DETECTION_SERVICE_PATH.exists(),
    all([
        len(missing_output_fields) == 0,
        risk_score_valid,
        ml_score_valid,
        decision_valid,
        rule_flags_valid,
        reason_valid,
        latency_valid
    ]),
    all([
        len(integration_results_df) == sample_size,
        batch_risk_valid,
        batch_ml_valid,
        batch_decision_valid,
        batch_reason_valid,
        batch_latency_valid
    ]),
    production_output_valid,
    production_batch_valid,
    INTEGRATION_VALIDATION_PATH.exists()
])

print()
print("================================================")
print("   STREAMSENTINEL V2 — PHASE 11 SUMMARY")
print("================================================")

print()

print(
    "ML features:",
    len(FEATURE_COLUMNS)
)

print(
    "Rule Engine available:",
    callable(evaluate_rules)
)

print(
    "ML Service available:",
    callable(get_ml_score)
)

print(
    "SHAP available:",
    callable(explain_transaction)
)

print(
    "Decision Engine available:",
    callable(make_decision)
)

print()

print(
    "Single transaction contract valid:",
    all([
        len(missing_output_fields) == 0,
        risk_score_valid,
        ml_score_valid,
        decision_valid,
        rule_flags_valid,
        reason_valid,
        latency_valid
    ])
)

print(
    "100 transaction integration valid:",
    all([
        len(integration_results_df) == sample_size,
        batch_risk_valid,
        batch_ml_valid,
        batch_decision_valid,
        batch_reason_valid,
        batch_latency_valid
    ])
)

print(
    "Production output valid:",
    production_output_valid
)

print(
    "Production 100 transaction test:",
    production_batch_valid
)

print(
    "Validation file saved:",
    INTEGRATION_VALIDATION_PATH.exists()
)

print()
print(
    "FINAL INTEGRATION STATUS:",
    "READY"
    if final_integration_ready
    else "NOT READY"
)

print(
    "Overall verification:",
    final_integration_ready
)

print("================================================")


   STREAMSENTINEL V2 — PHASE 11 SUMMARY

ML features: 6
Rule Engine available: True
ML Service available: True
SHAP available: True
Decision Engine available: True

Single transaction contract valid: True
100 transaction integration valid: True
Production output valid: True
Production 100 transaction test: True
Validation file saved: True

FINAL INTEGRATION STATUS: READY
Overall verification: True
